<a href="https://colab.research.google.com/github/cgcastroj/AgroSens/blob/master/Pr%C3%A1ctica_2_Copia_de_Reconocimiento_de_d%C3%ADgitos_en_tiempo_real_con_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datos de alumno
**Nombre:** Carlos Guillermo Castro Jiménez (20170629).

**Materia:** Inteligencia Artificial (07:00 - 08:00).

**Docente**: José Luis Medina Jiménez.

**Práctica:** Reconocimiento de dígitos en tiempo real con CNN.

# Introducción
## Objetivo de la práctica
El objetivo de la práctica es crear un sistema capaz de reconocer números escritos a mano.
En esta ocasión, una imagen debe de ser 28x28 pixeles y debe devolver una salida entre 0 y 9.

Para solucionarlo se debe utilizar Redes Neuronales Convulacionales (CNN)

---

# Preparación del modelo

## Importación de librerías


Importé librerías necesarias para el trabajo de redes neuronales y datos.

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import numpy as np


## Carga del dataset

Cargué el dataset MNIST  para entrenar el modelo de reconocimiento de dígitos.

In [ ]:
# Descarga del dataset MNIST.
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

A diferencia de la primer práctica, en esta ocasión el dataset se necesitó dividir en cuatro variables:
* train_images: Contiene las imágenes que se usarán para entrenar la red neuronal.
* train_labels: Contiene las etiquetas de cada imagen de entrenamiento.
* test_images: Contiene las imágenes que no serán utilizadas durante el entrenamiento.
* test_labels: Contiene las etiquetas correspondientes a las imágenes de prueba.

**Nota:** Las etiquetas son importantes porque indican qué número aparece en la imagen.

## Preprocesamiento

Después de cargar el dataset, los datos son preprocesados y adaptados al formato que la red neuronal necesita.



In [ ]:
# Formato de la imagen.
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# Normalización de los pixeles.
train_images = train_images / 255.0
test_images = test_images / 255.0

Las redes neuronales convolucionales requieren que las imágenes tengan un formato adicional que represente el número de canales de color.

El formato esperado de las imágenes siempre ha sido:


```
# (alto, ancho, canales)
```
Y, como aquí estamos trabajando con redes neuronales convolucionales, el formato cambia a:
```
# (imágenes, algo, ancho, canales de color)
```
Donde el valor "1" en el campo de "canales de color" significa que la imagen está en escala de grises.

El no normalizar provocaría que el modelo tardara más en aprender o puede tener problemas para funcionar, por lo considero como una buena práctica.


## Construcción de la Red Neuronal Convolucional (CNN)

Implementé la arquitectura del modelo donde se organizan las capas de la red neuronal.

**La arquitectura es:**

Imagen → Capa covolucional → Pooling → Convolución → Pooling → Clasificación

In [ ]:
model = models.Sequential()

# Primera capa convolucional.
model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)))

# Primera capa de pooling.
model.add(layers.MaxPooling2D((2,2)))

# Segunda capa convolucional.
model.add(layers.Conv2D(64, (3,3), activation='relu'))

# Segunda capa de pooling.
model.add(layers.MaxPooling2D((2,2)))

# Conversión de arrays bidimensionales a vectores.
model.add(layers.Flatten())

# Capa densa.
model.add(layers.Dense(64, activation='relu'))

# Capa de salida.
model.add(layers.Dense(10, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


El flujo de información es:
1. **Primera capa convolucional.**

   ```
   model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)))
   ```
   32 indica el número de filtros o detectores de patrones que tendrá cada capa.
   
   (3, 3) solo es el tamaño del filtro que se mueve por la imagen.

   activation='relu' transforma  los valores negativos a cero y mantiene los positivos.

   El resto de la instrucción (que es el input_shape(28,28,1)) ya lo habíamos descrito arriba.

2. **Primera capa de pooling.**

   ```
   model.add(layers.MaxPooling2D((2,2)))
   ```
   El pooling reduce el tamaño de la imagen procesada, en este caso, 2x2 significa que se toma el valor máximo dentro de cada bloque de 2x2 píxeles.

3. **Segunda capa convolucional.**

   ```
   model.add(layers.Conv2D(64, (3,3), activation='relu'))
   ```
   Ahora, en vez de 32 filtros, se usan 64 filtros. Esto lo hice porque la primer capa convolucional solamente detecta bordes y curvas, pero, en teoría, este incremento de filtros permitiría que sean detectados características más complejas.

4. **Segunda capa de pooling**

   ```
   model.add(layers.MaxPooling2D((2,2)))
   ```
   Simplemente se repite el proceso de reducción del tamaño de la imagen.

5. **Segunda capa de pooling**

   ```
   model.add(layers.Flatten())
   ```
   Como el procesamiento de imágenes contiene matrices bidimencionales, Flatten convierte todos esos valores en un solo vector.

0. **Capa oculta**

   ```
   model.add(layers.Dense(64, activation='relu'))
   ```
   Esto simplemente es un clasificador interno. La función "relu" permite que el modelo aprenda relaciones complejas entre las características de la imagen.

0. **Capa de salida**

   ```
   model.add(layers.Dense(10, activation='softmax'))
   ```
   Capa final del modelo.

   El modelo tiene 10 neuronas porque existen 10 posibles clases (0-9).








## Compilación del modelo

La compilación del modelo configura cómo la red aprenderá durante el entrenamiento.

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## Entrenar el modelo

En entrenamiento del modelo consiste en analizar miles de ejemplos de números (del dataset, por supuesto) y ajustar sus parámetros internos para aprender a reconocerlos.

El proceso  del entrenamiento es:
1. El modelo recibe una imagen (train_images, train_labels) del dataset.
2. Intenta predecir qué número aparece.
3. Se compara la predicción con la etiqueta correcta.
4. Se calcula el error.
5. El  optimizador ajusa los pesos de la red para reducir ese error.

Este proceso se repite durante 5 épocas.

In [ ]:
model.fit(train_images, train_labels, epochs=5, validation_data=(test_images, test_labels))

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 55s 28ms/step - accuracy: 0.9085 - loss: 0.3189 - val_accuracy: 0.9851 - val_loss: 0.0441
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 56s 30ms/step - accuracy: 0.9858 - loss: 0.0475 - val_accuracy: 0.9849 - val_loss: 0.0436
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 52s 28ms/step - accuracy: 0.9905 - loss: 0.0306 - val_accuracy: 0.9879 - val_loss: 0.0343
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 80s 27ms/step - accuracy: 0.9930 - loss: 0.0216 - val_accuracy: 0.9900 - val_loss: 0.0290
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.9952 - loss: 0.0157 - val_accuracy: 0.9894 - val_loss: 0.0301


## Guardar el modelo

Guardo el modelo entrenado. De esta manera, en la siguiente sección lo podré usar para las pruebas.

In [ ]:
model.save("modelo_mnist.keras")5

# Uso del modelo

## Librerías necesarias

In [ ]:
import cv2
from google.colab.output import eval_js
from base64 import b64decode

## Función para tomar la fotografía

In [ ]:
def take_photo(filename='photo.jpg', quality=0.8):
  js = """
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Tomar foto';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks()[0].stop();
      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }
  """

  data = eval_js(js + f'takePhoto({quality})')
  binary = b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(binary)

  return filename

## Tomar la fotografía

In [ ]:
filename = take_photo()

## Leer la imagen

In [ ]:
img = cv2.imread(filename)

## Procesamiento de imagen para MNIST

In [ ]:
img = cv2.resize(img, (28,28))
img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img = img / 255.0
img = img.reshape(1,28,28,1)

# Hacer la predicción

In [ ]:
prediction = model.predict(img)

digit = np.argmax(prediction)

print("Número detectado:", digit)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
Número detectado: 2


# Notas de la práctica

### ¿Qué es el dataset MNIST?
El dataset MNIST contiene 70,000 imágenes de números escritos a mano, donde cada imagen tiene un tamaño de 28x28 píxeles y una escala de grises de 0 a 255.
El dataset se distribuye en 60,000 imágenes para entrenamiento y 10,000 imágenes para prueba.

La razón por la que se usa MNIST en esta práctica es porque es pequeño y fácil de entrenar, es un dataset limpio y bien etiquetado y, además, es el estándar para comenzar a trabajar con redes neuronales.

Referencia: https://es.wikipedia.org/wiki/Base_de_datos_MNIST

### ¿Qué es CNN?
Una CNN (Convolutional Neural Network) está diseñada para detectar patrones especiales en imágenes. Su principal ventaja es que una CNN aprende automáticamente características visuales.

Las CNN funcionan con tres conceptos principales:
* **Convoluciones:** Detecta patrones locales como bordes o curvas.
* **Pooling:** Reduce la dimensionalidad de la imagen conservando información importante.
* **Capas densas:** Realizan la clasificación final.

